In [1]:
from dlrm_v3.configs import (
    get_embedding_table_config,
    get_hstu_configs,
)
from modules.dlrm_hstu import DlrmHSTU, DlrmHSTUConfig
from torchrec.modules.embedding_configs import EmbeddingConfig
from torchrec.modules.embedding_modules import (
    EmbeddingBagCollection,
    EmbeddingCollection,
)

In [2]:
dataset = 'msan-iterable'
hstu_config = get_hstu_configs(dataset)
table_config = get_embedding_table_config(dataset)

In [3]:
hstu_config

DlrmHSTUConfig(max_seq_len=2056, max_num_candidates=1, max_num_candidates_inference=5, hstu_num_heads=4, hstu_attn_linear_dim=128, hstu_attn_qk_dim=128, hstu_attn_num_layers=3, hstu_embedding_table_dim=256, hstu_transducer_embedding_dim=512, hstu_group_norm=True, hstu_input_dropout_ratio=0.2, hstu_linear_dropout_rate=0.1, contextual_feature_to_max_length={}, contextual_feature_to_min_uih_length={}, candidates_weight_feature_name='item_action_weight', candidates_watchtime_feature_name='item_target_watchtime', causal_multitask_weights=0.2, multitask_configs=[TaskConfig(task_name='is_relevant', task_weight=1, task_type=<MultitaskTaskType.BINARY_CLASSIFICATION: 0>)], user_embedding_feature_names=['event_id'], item_embedding_feature_names=['item_event_id'], uih_post_id_feature_name='event_id', uih_action_time_feature_name='action_timestamp', uih_weight_feature_name='action_weight', hstu_uih_feature_names=['event_id', 'action_timestamp', 'action_weight', 'watch_time'], hstu_candidate_feature

In [4]:
table_config

{'event_id': EmbeddingConfig(num_embeddings=10000000, embedding_dim=256, name='event_id', data_type=<DataType.FP16: 'FP16'>, feature_names=['event_id', 'item_event_id'], weight_init_max=None, weight_init_min=None, num_embeddings_post_pruning=None, init_fn=functools.partial(<function uniform_ at 0x7f80e190ef80>, a=-0.00031622776601683794, b=0.00031622776601683794), need_pos=False)}

In [5]:
from modules.embedding_modules import PrecomputedEmbeddingModule
from dlrm_v3.train.utils import make_model

In [6]:
import os

base_mount_path = '/scratch/azureml/cr/j/55428fd652604523af0a40ae36ce2d22/cap/data-capability/wd/INPUT_data'

emb_path_5m = os.path.join(base_mount_path, 'EventEmb.npy')
seq_path_5m = os.path.join(base_mount_path, 'Behaviors.tsv')
emb_path_358k = os.path.join(base_mount_path, 'EventEmb_358k.npy')
seq_path_358k = os.path.join(base_mount_path, 'Behaviors_358k.tsv')

In [24]:
import os
import numpy as np
import torch
import abc

class EmbeddingModule(torch.nn.Module):
    @abc.abstractmethod
    def debug_str(self) -> str:
        pass

    @abc.abstractmethod
    def get_item_embeddings(self, item_ids: torch.Tensor) -> torch.Tensor:
        pass

    @property
    @abc.abstractmethod
    def item_embedding_dim(self) -> int:
        pass

# class PrecomputedEmbeddingModule(torch.nn.Module):
class PrecomputedEmbeddingModule(EmbeddingModule):
    def __init__(
        self, 
        emb_file_path: str,  # Single .npy file path
        output_dim: int = 256,  # hstu_embedding_table_dim
        preload: bool = False
    ) -> None:
        super().__init__()
        self.emb_file_path = emb_file_path
        self.output_dim = output_dim
        
        # Load entire file using memory mapping
        self.embedding_array = np.load(emb_file_path, mmap_mode="r")
        
        # Auto-detect input_dim from npy file shape
        self.input_dim = self.embedding_array.shape[1]
        
        print(f"Loaded embedding file: {emb_file_path}")
        print(f"Shape: {self.embedding_array.shape}, input_dim: {self.input_dim}, output_dim: {output_dim}")
        print(f"Embedding dtype: {self.embedding_array.dtype}")
        
        # Projection layer: from precomputed dim to target dim
        self.proj = torch.nn.Linear(self.input_dim, output_dim, dtype=torch.float32)
        

    def get_item_embeddings(self, item_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            item_ids: Tensor of shape [K], item IDs (starting from 1)
        Returns:
            Tensor of shape [K, output_dim]
        """


        # emb_cpu = self.embedding_array(item_ids-1)
        # emb = emb_cpu.to(device)
        # return self.proj(emb)

        device = item_ids.device
        
        # Convert to numpy indices (subtract 1 for 0-based indexing)
        indices = (item_ids - 1).cpu().numpy()
        
        # Index the memory-mapped array
        emb_numpy = self.embedding_array[indices]
        
        # Convert to PyTorch tensor and move to device
        emb_cpu = torch.from_numpy(emb_numpy).float()
        emb = emb_cpu.to(device)

        return self.proj(emb)
        
        # original_shape = item_ids.shape
        # flat_ids = item_ids.view(-1)
        # device = flat_ids.device

        # K = flat_ids.size(0)

        # embeddings = [None] * K

        # for i, item_id in enumerate(flat_ids):
        #     item_ids_np = (item_id - 1).cpu().numpy().astype(np.int64)
        #     embeddings_np = self.embedding_array[item_ids_np]  # shape: [K, input_dim]
        #     emb_cpu = torch.from_numpy(embeddings_np).float()  # Convert to float32 tensor
        #     embeddings[i] = emb_cpu.to(device)  # Move to target device

        # # Stack embeddings into a single tensor
        # stacked_embeddings = torch.stack(embeddings, dim=0).view(*original_shape, -1)  # shape: [B, N, D]
        # return self.proj(stacked_embeddings)  # Project to output_dim
    

        # embeddings = embeddings.view(*original_shape, self.input_dim).to(device)

        # # Ensure projection layer is on the same device as input
        # if self.proj.weight.device != device:
        #     self.proj = self.proj.to(device)
        
        # # Project to target dimension
        # return self.proj(embeddings)

    def debug_str(self) -> str:
        return f"precomputed_emb_file_{os.path.basename(self.emb_file_path)}"

In [8]:
import torch

encoded_ids = torch.tensor([1, 2, 3, 4])

In [25]:
model = PrecomputedEmbeddingModule(emb_file_path = emb_path_5m)

Loaded embedding file: /scratch/azureml/cr/j/55428fd652604523af0a40ae36ce2d22/cap/data-capability/wd/INPUT_data/EventEmb.npy
Shape: (366223689, 64), input_dim: 64, output_dim: 256
Embedding dtype: float64


In [26]:
output = model.get_item_embeddings(encoded_ids)

In [39]:
print("Output shape:", output.shape)
print("Output sample:", output)

Output shape: torch.Size([4, 256])
Output sample: tensor([[ 0.1914,  0.2880, -0.0824,  ..., -0.0032, -0.1922, -0.1224],
        [-0.0048,  0.1172, -0.2595,  ..., -0.0909,  0.1217, -0.0768],
        [ 0.0789, -0.0927, -0.2905,  ..., -0.0608,  0.0610, -0.0990],
        [ 0.0215, -0.0916, -0.2945,  ...,  0.0636, -0.0490, -0.2639]],
       grad_fn=<AddmmBackward0>)


In [27]:
print("Output shape:", output.shape)
print("Output sample:", output)

Output shape: torch.Size([4, 256])
Output sample: tensor([[-0.0674, -0.1221, -0.2345,  ...,  0.0409,  0.0887,  0.0680],
        [-0.1271, -0.0039,  0.0182,  ..., -0.1845, -0.0597, -0.0281],
        [-0.2392, -0.0352, -0.0454,  ...,  0.0257,  0.0039, -0.0719],
        [-0.1837, -0.1020, -0.1725,  ..., -0.1192,  0.1648, -0.0247]],
       grad_fn=<AddmmBackward0>)


In [14]:
embedding_array = np.load(emb_path_5m, mmap_mode="r")

In [19]:
embedding_array[:5]

memmap([[-0.14902773, -0.31867605, -0.48099414, -0.14001867,  0.166962  ,
         -0.27996734, -0.39512065,  0.351032  ,  0.18013343,  0.6449606 ,
         -0.24878846,  0.28966227, -0.1165188 ,  0.57290787, -0.03482784,
          0.03895671, -0.05957981, -0.0216736 , -0.21384196, -0.22438331,
          0.18114817,  0.38757968, -0.16088326, -0.2535081 ,  0.26060098,
         -0.20885217,  0.00553665, -0.12122772,  0.32043025, -0.16423331,
          0.3399551 ,  0.05719351, -0.3094798 ,  0.01602199,  0.0815412 ,
          0.32996312, -0.3936962 , -0.20090717,  0.17033018, -0.20758943,
          0.14197089,  0.150404  ,  0.2399098 ,  0.20193069, -0.17422609,
         -0.16565518,  0.3564529 ,  0.25678077, -0.25523648, -0.88576984,
          0.4727903 , -0.08600264, -0.09532001,  0.15484275, -0.08467265,
          0.27088824,  0.07099652, -0.05364101,  0.10646308,  0.10453409,
         -0.45578772,  0.08718601,  0.14157657, -0.6214159 ],
        [-0.19081563, -0.12149302,  0.12215967,  0

In [5]:
import os
cpu_count = os.cpu_count()  # 例如：32 核
cpu_count

40

In [7]:
world_size = 8
recommended_workers = cpu_count // world_size
recommended_workers

5